### Step 0: Install unsloth package

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install openpyxl

### Step 1: Generate jsonl data from XLSX

In [ ]:
!python prepare_data.py

Reading data/raw/DataCases.xlsx...

Splitting data (dev_size=0.1, test_size=0.1, random_state=42)...
  Train set: 244 examples (79.7%)
  Dev set: 31 examples (10.1%)
  Test set: 31 examples (10.1%)

Writing data/processed/train.jsonl...
✓ Created data/processed/train.jsonl (244 examples)
Writing data/processed/dev.jsonl...
✓ Created data/processed/dev.jsonl (31 examples)
Writing data/processed/test.jsonl...
✓ Created data/processed/test.jsonl (31 examples)
✓ Conversion complete!


### Step 2: Load the raw data and add instruction formatting

In [ ]:
import os
import re
import json
from datasets import load_dataset, Dataset

PATH = "data/processed"
train_url = f"{PATH}/train.jsonl"
dev_url = f"{PATH}/dev.jsonl"
test_url = f"{PATH}/test.jsonl"

# Instruction template for legal case analysis
INSTRUCTION = '''You are a legal case analyst. Given a legal case document, you must produce a structured IRAC summary.

### INPUT FORMAT:
You will receive:
- **[Case Name]**: The name/title of the legal case
- **[Case]**: The full text of the legal case document

### OUTPUT FORMAT:
Produce a structured analysis with the following sections:

**Case Name:** [Repeat the case name]

**Court:** [Identify the court from the case text]

**Jurisdiction:** [Identify the jurisdiction (e.g., State, Federal, Circuit)]

**Area of Law:** [Identify the primary legal area (e.g., Contract Law, Tort Law, Criminal Law, Constitutional Law)]

**Procedural Posture:** [State the procedural stage (e.g., "Appeal from summary judgment", "Trial court decision", "Motion to dismiss")]

**I. Issue**
Frame the central legal question in "whether...when..." format. Include: (1) the parties by role, (2) the legal doctrine at issue, and (3) the key factual circumstances. Keep to 1-2 sentences.

**II. Rule**
State the controlling legal rule, test, or standard the court applied. Identify the source of law (statute, regulation, constitutional provision, or precedent). Present multi-part tests in element or step form. Include any exceptions or limitations. Use 1-2 short paragraphs with citations to the judgment where applicable.

**III. Application**
Explain how the court applied the rule to the material facts. Walk through each element or step of the rule. Use connecting language ("because", "since", "given that") to link law and facts. For each key element: state it, identify the relevant facts, explain the court's reasoning, and note any precedent used. This should be the longest section, focusing on the contested elements and the factual basis for the decision.

**IV. Conclusion**
State the court's bottom-line legal holding that answers the Issue. Explicitly state: "The court held that [legal conclusion], therefore [legal consequence]." Include the procedural disposition (e.g., "appeal allowed", "appeal dismissed", "motion granted"). Keep to 1-2 sentences.

### EXAMPLE:
If you receive:
[Case Name] Smith v. Jones
[Case] [full case text here...]

You should output:
**Case Name:** Smith v. Jones
**Court:** Supreme Court of California
**Jurisdiction:** California
**Area of Law:** Contract Law
**Procedural Posture:** Appeal from summary judgment

**I. Issue**
[Your analysis...]

**II. Rule**
[Your analysis...]

**III. Application**
[Your analysis...]

**IV. Conclusion**
[Your analysis...]

---

Input:
'''


def format_example(example):
    """
    Format a raw example into instruction format for training.
    
    Args:
        example: dict with id, case_name, summary, IRAC_summ
    
    Returns:
        dict with id and text (formatted for training)
    """
    case_input = f"[Case Name] {example['case_name']}\n[Case] {example['case']}"
    prompt = INSTRUCTION + case_input + "\n\nOutput:"
    
    return {
        "id": example["id"],
        "text": f"<|user|>\n{prompt}\n<|assistant|>\n{example['IRAC_summ']}"
    }


# Load raw train and dev data
raw_train_dataset = load_dataset("json", data_files=train_url)
raw_dev_dataset = load_dataset("json", data_files=dev_url)

# Apply instruction formatting
train_dataset = raw_train_dataset["train"].map(format_example)
dev_dataset = raw_dev_dataset["train"].map(format_example)  # Note: load_dataset puts single files under "train" key

print(f"Loaded and formatted {len(train_dataset)} training examples")
print(f"Loaded and formatted {len(dev_dataset)} dev examples")

Loaded and formatted 244 training examples
Loaded and formatted 31 dev examples


### Display the data info

In [ ]:
print(f"Dataset size: {len(train_dataset)}")
print(f"\nRaw fields: {raw_train_dataset['train'].column_names}")
print(f"Formatted fields: {train_dataset.column_names}")
print(f"\n--- Sample formatted text (first 500 chars) ---")
print(train_dataset["text"][0])

Dataset size: 244

Raw fields: ['id', 'case_name', 'case', 'IRAC_summ']
Formatted fields: ['id', 'case_name', 'case', 'IRAC_summ', 'text']

--- Sample formatted text (first 500 chars) ---
<|user|>
You are a legal case analyst. Given a legal case document, you must produce a structured IRAC summary.

### INPUT FORMAT:
You will receive:
- **[Case Name]**: The name/title of the legal case
- **[Case]**: The full text of the legal case document

### OUTPUT FORMAT:
Produce a structured analysis with the following sections:

**Case Name:** [Repeat the case name]

**Court:** [Identify the court from the case text]

**Jurisdiction:** [Identify the jurisdiction (e.g., State, Federal, Circuit)]

**Area of Law:** [Identify the primary legal area (e.g., Contract Law, Tort Law, Criminal Law, Constitutional Law)]

**Procedural Posture:** [State the procedural stage (e.g., "Appeal from summary judgment", "Trial court decision", "Motion to dismiss")]

**I. Issue**
Frame the central legal question in "w

### Step 4: Load the LLM from Hugging Face and apply LoRA for fine-tuning


In [ ]:
from unsloth import FastLanguageModel

# unsloth/Qwen3-14B-unsloth-bnb-4bit
model_id = "unsloth/Qwen3-14B-unsloth-bnb-4bit"

# tokenizer and model setting
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 11000,
    load_in_4bit = True,
)

# lora setting
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L40S. Num GPUs = 1. Max memory: 44.638 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.1.4 patched 40 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


### Step 5: Start LoRA Fine-Tuning


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = dev_dataset,
    dataset_text_field = "text",
    max_seq_length = 11000,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2,
        warmup_steps = 20,
        num_train_epochs = 6, # epochs
        learning_rate = 1e-4, # learning rate
        logging_steps = 50,
        save_steps = 400,
        eval_steps = 50,  # Evaluate every 400 steps
        eval_strategy = "steps",  # Evaluate at every eval_steps
        save_strategy = "steps",  # Save at every save_steps
        load_best_model_at_end = True,  # Load the best model at the end
        metric_for_best_model = "eval_loss",  # Metric to determine best model
        greater_is_better = False,  # Lower loss is better
        save_total_limit = 3,  # Only keep the 3 best checkpoints
        fp16 = False,
        bf16 = True,
        report_to = "none",
        output_dir = "./Qwen3-14B", # save Lora
    ),
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 244 | Num Epochs = 6 | Total steps = 366
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 20,971,520 of 14,789,278,720 (0.14% trained)


Step,Training Loss,Validation Loss
50,1.190100,0.640061
100,0.551500,0.538917
150,0.480300,0.516050
200,0.471000,0.509442
250,0.436700,0.502231
300,0.426800,0.500961
350,0.409600,0.503687


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=366, training_loss=0.5599227055825823, metrics={'train_runtime': 611.915, 'train_samples_per_second': 2.392, 'train_steps_per_second': 0.598, 'total_flos': 1.260220965387264e+17, 'train_loss': 0.5599227055825823, 'epoch': 6.0})

### Step 6: Run Inference and Extract Structured Outputs

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
import json
from rouge_score import rouge_scorer
from tqdm import tqdm
import torch
import os

# Load raw test data and format it (using same INSTRUCTION from Step 2)
raw_test_dataset = load_dataset("json", data_files=test_url)

# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)


def format_test_prompt(example):
    """Format raw test example into prompt (without the answer)."""
    case_input = f"[Case Name] {example['case_name']}\n[Case] {example['case']}"
    prompt = INSTRUCTION + case_input + "\n\nOutput:"
    return prompt


def extract_irac_summary(decoded_text):
    """
    Extract the IRAC summary from the model's decoded output.
    Handles various model output formats.
    """
    result = decoded_text

    # Try to split by <|im_start|>assistant format (Qwen/ChatML style)
    if "<|im_start|>assistant" in result:
        parts = result.split("<|im_start|>assistant")
        if len(parts) > 1:
            result = parts[-1].strip()

    # Try to split by <|assistant|> format
    elif "<|assistant|>" in result:
        parts = result.split("<|assistant|>")
        if len(parts) > 1:
            result = parts[-1].strip()

    # If no delimiter found, try to extract everything after "Output:"
    elif "Output:" in result:
        parts = result.split("Output:")
        if len(parts) > 1:
            result = parts[-1].strip()

    # Remove end tokens
    end_tokens = ["<|im_end|>", "<|endoftext|>", "</s>", "<|end|>", "<|eot_id|>"]
    for token in end_tokens:
        result = result.replace(token, "")

    # Clean up any remaining special tokens
    result = re.sub(r'<\|[^|]+\|>', '', result)

    result = result.strip()

    # Fallback: return last substantial part if result is empty
    if not result:
        lines = decoded_text.split("\n")
        meaningful_lines = [line for line in lines if line.strip() and not line.strip().startswith("<|")]
        if meaningful_lines:
            return "\n".join(meaningful_lines[-5:])

    return result


# Define output path and create directory if it doesn't exist
output_path = "results/Qwen3-14B-11000"
os.makedirs(output_path, exist_ok=True)
print(f"Output directory: {output_path}")

# Perform inference
results = []
all_rouge_scores = []
print("Starting inference on legal cases...")
print(f"Loading data from: {test_url}")
print(f"Total test samples: {len(raw_test_dataset['train'])}")
print("=" * 60)

# Use tqdm for progress bar
for i, sample in enumerate(tqdm(raw_test_dataset["train"], desc="Processing cases")):
    
    case_name = sample["case_name"]
    ground_truth = sample["IRAC_summ"]

    # Format the prompt using the instruction template
    user_content = format_test_prompt(sample)
    messages = [{"role": "user", "content": user_content}]

    # Apply chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    
    # Generate prediction with MUCH smaller max_new_tokens
    with torch.no_grad():  # Disable gradient computation for faster inference
        result = model.generate(
            **inputs,
            max_new_tokens=1024,  # Reduced from 8192 to 1024 (still ~700 words)
            temperature=0.7,
            top_p=0.9,  # Slightly higher for better quality
            top_k=50,   # Increased for more diversity
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode the result
    decoded = tokenizer.decode(result[0], skip_special_tokens=False)

    # Extract IRAC summary from output
    irac_summary = extract_irac_summary(decoded)

    # Prepare output data
    dump_data = {
        "id": sample.get("id", f"case_{i + 1}"),
        "case_name": case_name,
        "predicted_IRAC": irac_summary,
        "ground_truth_IRAC": ground_truth,
    }

    # Compute ROUGE scores
    scores = scorer.score(ground_truth, irac_summary)
    rouge_scores = {
        "rouge1_precision": scores['rouge1'].precision,
        "rouge1_recall": scores['rouge1'].recall,
        "rouge1_fmeasure": scores['rouge1'].fmeasure,
        "rouge2_precision": scores['rouge2'].precision,
        "rouge2_recall": scores['rouge2'].recall,
        "rouge2_fmeasure": scores['rouge2'].fmeasure,
        "rougeL_precision": scores['rougeL'].precision,
        "rougeL_recall": scores['rougeL'].recall,
        "rougeL_fmeasure": scores['rougeL'].fmeasure,
    }
    dump_data["rouge_scores"] = rouge_scores
    all_rouge_scores.append(rouge_scores)

    # Print progress every 10 samples
    if (i + 1) % 10 == 0:
        print(f"\nProcessed {i + 1}/{len(raw_test_dataset['train'])} cases")
        print(f"Latest - {case_name}")
        print(f"ROUGE-1: {scores['rouge1'].fmeasure:.4f} | ROUGE-2: {scores['rouge2'].fmeasure:.4f} | ROUGE-L: {scores['rougeL'].fmeasure:.4f}")

    results.append(dump_data)

print("\n" + "=" * 60)
print(f"✓ Inference complete! Processed {len(results)} cases.")

# Save results to JSON file
output_file = os.path.join(output_path, "legal_cases_predictions.json")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✓ Results saved to {output_file}")

# Also save as JSONL for easy processing
output_jsonl = os.path.join(output_path, "legal_cases_predictions.jsonl")
with open(output_jsonl, 'w', encoding='utf-8') as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"✓ Results also saved to {output_jsonl}")

# Print summary statistics
print("\n" + "=" * 60)
print("Summary:")
print(f"  Total cases processed: {len(results)}")
print(f"  Average prediction length: {sum(len(r['predicted_IRAC']) for r in results) / len(results):.0f} chars")

# Compute and print average ROUGE scores
if all_rouge_scores:
    avg_rouge = {
        "rouge1_fmeasure": sum(s["rouge1_fmeasure"] for s in all_rouge_scores) / len(all_rouge_scores),
        "rouge2_fmeasure": sum(s["rouge2_fmeasure"] for s in all_rouge_scores) / len(all_rouge_scores),
        "rougeL_fmeasure": sum(s["rougeL_fmeasure"] for s in all_rouge_scores) / len(all_rouge_scores),
        "rouge1_precision": sum(s["rouge1_precision"] for s in all_rouge_scores) / len(all_rouge_scores),
        "rouge1_recall": sum(s["rouge1_recall"] for s in all_rouge_scores) / len(all_rouge_scores),
        "rouge2_precision": sum(s["rouge2_precision"] for s in all_rouge_scores) / len(all_rouge_scores),
        "rouge2_recall": sum(s["rouge2_recall"] for s in all_rouge_scores) / len(all_rouge_scores),
        "rougeL_precision": sum(s["rougeL_precision"] for s in all_rouge_scores) / len(all_rouge_scores),
        "rougeL_recall": sum(s["rougeL_recall"] for s in all_rouge_scores) / len(all_rouge_scores),
    }

    print("\n" + "-" * 60)
    print("Average ROUGE Scores:")
    print("-" * 60)
    print(f"  ROUGE-1:  P={avg_rouge['rouge1_precision']:.4f}  R={avg_rouge['rouge1_recall']:.4f}  F1={avg_rouge['rouge1_fmeasure']:.4f}")
    print(f"  ROUGE-2:  P={avg_rouge['rouge2_precision']:.4f}  R={avg_rouge['rouge2_recall']:.4f}  F1={avg_rouge['rouge2_fmeasure']:.4f}")
    print(f"  ROUGE-L:  P={avg_rouge['rougeL_precision']:.4f}  R={avg_rouge['rougeL_recall']:.4f}  F1={avg_rouge['rougeL_fmeasure']:.4f}")

    # Save ROUGE summary to a separate file
    rouge_summary = {
        "num_samples": len(all_rouge_scores),
        "average_scores": avg_rouge,
        "per_sample_scores": all_rouge_scores
    }
    rouge_output_file = os.path.join(output_path, "rouge_scores.json")
    with open(rouge_output_file, 'w', encoding='utf-8') as f:
        json.dump(rouge_summary, f, indent=2)
    print(f"\n✓ ROUGE scores saved to {rouge_output_file}")

print("=" * 60)
print(f"\nAll results saved to: {output_path}/")
print(f"  - legal_cases_predictions.json")
print(f"  - legal_cases_predictions.jsonl")
print(f"  - rouge_scores.json")

NameError: name 'load_dataset' is not defined

In [ ]:
# ============================================================================
# Simple ROUGE Evaluation Cell
# ============================================================================
# Usage: Just set the prediction_file variable and run this cell

import json
from rouge_score import rouge_scorer
import numpy as np

# ============================================================================
# CONFIGURATION - Change this to your prediction file
# ============================================================================
prediction_file = "/teamspace/studios/this_studio/results/Qwen3-14B-11000/legal_cases_predictions.json"  # <-- Change this to your file

# ============================================================================
# Load predictions
# ============================================================================
print(f"Loading predictions from: {prediction_file}")
with open(prediction_file, 'r', encoding='utf-8') as f:
    predictions = json.load(f)

print(f"✓ Loaded {len(predictions)} predictions\n")

# ============================================================================
# Compute ROUGE scores
# ============================================================================
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_scores = []
for pred in predictions:
    scores = scorer.score(pred["ground_truth_IRAC"], pred["predicted_IRAC"])
    rouge_scores.append({
        "rouge1": scores['rouge1'].fmeasure,
        "rouge2": scores['rouge2'].fmeasure,
        "rougeL": scores['rougeL'].fmeasure,
    })

# ============================================================================
# Print results
# ============================================================================
print("=" * 60)
print("ROUGE SCORES")
print("=" * 60)
print(f"\nSamples: {len(rouge_scores)}")
print(f"\nROUGE-1 F1: {np.mean([s['rouge1'] for s in rouge_scores]):.4f} (±{np.std([s['rouge1'] for s in rouge_scores]):.4f})")
print(f"ROUGE-2 F1: {np.mean([s['rouge2'] for s in rouge_scores]):.4f} (±{np.std([s['rouge2'] for s in rouge_scores]):.4f})")
print(f"ROUGE-L F1: {np.mean([s['rougeL'] for s in rouge_scores]):.4f} (±{np.std([s['rougeL'] for s in rouge_scores]):.4f})")
print("=" * 60)

Loading predictions from: /teamspace/studios/this_studio/results/Qwen3-14B-11000/legal_cases_predictions.json
✓ Loaded 31 predictions

ROUGE SCORES

Samples: 31

ROUGE-1 F1: 0.6120 (±0.0546)
ROUGE-2 F1: 0.3216 (±0.0512)
ROUGE-L F1: 0.3498 (±0.0429)
